# Lab 03.2 – XADC Supply Voltage Monitoring

## Objectives
- Identify the internal voltage channels exposed by the XADC Linux IIO driver.
- Read the raw ADC values for the Zynq supply rails.
- Convert raw values to millivolts and volts using the IIO scale information.
- Create reusable Python functions for voltage monitoring.

## Hardware
- Digilent Zybo legacy board
- PYNQ 3.0.1
- No external sensor or additional hardware is required.

## Background
Besides die temperature, the Zynq-7000 XADC monitors several internal supply rails. On this PYNQ image the measurements are available through Linux Industrial I/O (IIO).

The IIO convention for a voltage channel is:

\[
V_{mV} = (raw + offset) \times scale
\]

and therefore:

\[
V_V = V_{mV}/1000
\]

The driver may expose one scale file shared by several channels or a scale file specific to a channel. The code below handles both cases.


## Exercise 1 – Locate the XADC and list voltage channels


In [ ]:
from pathlib import Path
import re

iio_root = Path("/sys/bus/iio/devices")
xadc_path = None

for dev in sorted(iio_root.glob("iio:device*")):
    name_file = dev / "name"
    if name_file.exists() and name_file.read_text().strip() == "xadc":
        xadc_path = dev
        break

if xadc_path is None:
    raise RuntimeError("XADC IIO device not found.")

voltage_raw_files = sorted(xadc_path.glob("in_voltage*_raw"))

print("XADC device:", xadc_path)
print(f"Voltage channels found: {len(voltage_raw_files)}")
for f in voltage_raw_files:
    print(" ", f.name)


Typical internal channels include `vccint`, `vccaux`, `vccbram`, processor-system supply rails, and the XADC reference voltages. The exact list is determined by the Linux driver and device configuration.


## Exercise 2 – Find the IIO conversion parameters

The helper below searches for the most specific available `scale` or `offset` file and falls back to a shared value when required.


In [ ]:
def parameter_file(raw_file, parameter):
    name = raw_file.name
    if not name.endswith("_raw"):
        raise ValueError("Expected an IIO *_raw file.")

    base = name[:-4]
    candidates = [xadc_path / f"{base}_{parameter}"]

    m = re.match(r"^(in_voltage\d+)(?:_[A-Za-z0-9]+)?$", base)
    if m:
        candidates.append(xadc_path / f"{m.group(1)}_{parameter}")

    candidates.append(xadc_path / f"in_voltage_{parameter}")

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None

for raw_file in voltage_raw_files:
    scale_file = parameter_file(raw_file, "scale")
    offset_file = parameter_file(raw_file, "offset")
    print(raw_file.name)
    print("  scale :", scale_file.name if scale_file else "not found")
    print("  offset:", offset_file.name if offset_file else "not used")


## Exercise 3 – Read all internal voltages


In [ ]:
def channel_label(raw_file):
    base = raw_file.name[:-4]
    parts = base.split("_", 2)
    return parts[2] if len(parts) == 3 else base

def read_voltage(raw_file):
    raw = float(raw_file.read_text().strip())

    scale_file = parameter_file(raw_file, "scale")
    if scale_file is None:
        raise RuntimeError(f"No scale found for {raw_file.name}")
    scale = float(scale_file.read_text().strip())

    offset_file = parameter_file(raw_file, "offset")
    offset = float(offset_file.read_text().strip()) if offset_file else 0.0

    millivolts = (raw + offset) * scale
    return millivolts / 1000.0

print(f"{'Channel':<14} {'Voltage (V)':>12}")
print("-" * 27)

for raw_file in voltage_raw_files:
    label = channel_label(raw_file)
    voltage = read_voltage(raw_file)
    print(f"{label:<14} {voltage:>12.4f}")


## Exercise 4 – Monitor one supply rail repeatedly

The following code selects `vccint` when available. Otherwise it uses the first voltage channel found.


In [ ]:
import time

selected = next((f for f in voltage_raw_files if "vccint" in f.name), None)
if selected is None:
    if not voltage_raw_files:
        raise RuntimeError("No XADC voltage channels found.")
    selected = voltage_raw_files[0]

label = channel_label(selected)
print("Selected channel:", label)

for i in range(5):
    voltage = read_voltage(selected)
    print(f"Measurement {i + 1}: {voltage:.4f} V")
    time.sleep(1)


## Questions
1. Which internal voltage channels are exposed on your Zybo?
2. Why should the IIO `scale` value be used instead of assuming a fixed ADC conversion factor in Python?
3. Which channel represents the Zynq core supply voltage?
4. Why can repeated measurements of a nominally constant supply differ slightly?
5. Why are the XADC reference channels useful when evaluating ADC measurements?
